# Groundwater

In [92]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Load data

In [93]:
!pip install -q xgboost

import torch
print("GPU available:", torch.cuda.is_available())

GPU available: True


In [94]:
gw_df = pd.read_csv("groundwater_model_dataset.csv")

print("Dataset shape:", gw_df.shape)

print("\nColumns:")
print(gw_df.columns.tolist())

Dataset shape: (104090, 23)

Columns:
['Data Acquisition Time', 'District LGD Code', 'District', 'rainfall', 'groundwater_level', 'year', 'month', 'day', 'hour', 'day_of_week', 'season', 'rainfall_lag_1', 'rainfall_lag_3', 'rainfall_lag_6', 'groundwater_lag_1', 'groundwater_lag_3', 'groundwater_lag_6', 'rainfall_roll_3', 'rainfall_roll_6', 'groundwater_roll_3', 'groundwater_roll_6', 'rainfall_diff_1', 'groundwater_diff_1']


In [95]:
X = gw_df.drop(columns=["groundwater_level", "Data Acquisition Time"])
y = gw_df["groundwater_level"]

In [96]:
leak_cols = [
    "groundwater_roll_3", "groundwater_roll_6",
    "groundwater_lag_1", "groundwater_lag_3",
    "groundwater_lag_6"
]

X = X.drop(columns=leak_cols)

In [97]:
categorical_cols = ["District", "season"]
numerical_cols = [col for col in X.columns if col not in categorical_cols]

print("\nCategorical Columns:", categorical_cols)
print("Numerical Columns:", numerical_cols)



Categorical Columns: ['District', 'season']
Numerical Columns: ['District LGD Code', 'rainfall', 'year', 'month', 'day', 'hour', 'day_of_week', 'rainfall_lag_1', 'rainfall_lag_3', 'rainfall_lag_6', 'rainfall_roll_3', 'rainfall_roll_6', 'rainfall_diff_1', 'groundwater_diff_1']


# Preprocessor

In [98]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

le = LabelEncoder()
y_train = le.fit_transform(y_train)

# Train data split

In [99]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (83272, 16)
Test shape: (20818, 16)


# Modelling

In [100]:
models = {
    "Linear Regression": LinearRegression(),

    "Random Forest (CPU fast)": RandomForestRegressor(
        n_estimators=100,
        n_jobs=-1,
        random_state=42
    ),

    "XGBoost (T4 GPU)": XGBRegressor(
    tree_method="hist",
    device="cuda",
    n_estimators=300,
    max_depth=6,
    predictor="gpu_predictor",
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
}

In [ ]:
!nvidia-smi

Sun Apr  5 10:24:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P0             27W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [101]:
corr = df.corr(numeric_only=True)

print("\nCorrelation with flood_risk_score:")
print(corr["flood_risk_score"].sort_values(ascending=False))


Correlation with flood_risk_score:
flood_risk_score      1.000000
river_water_level     0.903284
river_roll_3          0.899019
river_lag_1           0.894575
river_roll_6          0.893620
river_lag_3           0.883525
year                  0.649152
groundwater_level     0.476478
groundwater_roll_3    0.473809
groundwater_roll_6    0.470003
groundwater_lag_1     0.469284
groundwater_lag_3     0.460428
District LGD Code     0.229947
river_diff_1          0.061557
rainfall_diff_1       0.053500
hour                  0.052875
day_of_week           0.046026
rainfall              0.038507
groundwater_diff_1    0.037802
day                  -0.008673
rainfall_roll_3      -0.008871
rainfall_lag_1       -0.020358
rainfall_roll_6      -0.043744
rainfall_lag_3       -0.044291
month                -0.227775
Name: flood_risk_score, dtype: float64


In [103]:
results = []
for name, model in models.items():
  print(f"\n===== {name} =====")
  pipeline = Pipeline([
    ("preprocessor", preprocessor), ("model", model)
  ])
  start = time.time()
  pipeline.fit(X_train, y_train)
  train_time = time.time() - start
  y_pred = pipeline.predict(X_test)
  mae = mean_absolute_error(y_test, y_pred)
  rmse = np.sqrt(mean_squared_error(y_test, y_pred))
  r2 = r2_score(y_test, y_pred)
  results.append([name, mae, rmse, r2, train_time])
  print(f"Training Time: {train_time:.2f}) sec")
  print("MAE:", mae)
  print("R²:", r2)
  print("RMSE:", rmse)


===== Linear Regression =====
Training Time: 1.07) sec
MAE: 7.174406338247941
R²: 0.3229014529905686
RMSE: 26.59248087295854

===== Random Forest (CPU fast) =====
Training Time: 1094.97) sec
MAE: 0.6757480330370618
R²: 0.965072368458264
RMSE: 6.039729425988344

===== XGBoost (T4 GPU) =====


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [12:09:08] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Training Time: 0.87) sec
MAE: 2.364097825048385
R²: 0.9498167933054464
RMSE: 7.239559123160559


In [ ]:
import seaborn as sns

corr = gw_df.corr(numeric_only=True)
print(corr["groundwater_level"].sort_values(ascending=False))

In [ ]:
results_df = pd.DataFrame(
    results,
    columns=["Model", "MAE", "RMSE", "R2 Score", "Training Time (s)"]
)

results_df = results_df.sort_values("R2 Score", ascending=False)

print("\n===== FINAL REGRESSION RESULTS =====")
print(results_df)

results_df.to_csv("groundwater_regression_results.csv", index=False)
print("\nSaved results as groundwater_regression_results.csv")


===== FINAL REGRESSION RESULTS =====
                      Model       MAE       RMSE  R2 Score  Training Time (s)
1  Random Forest (CPU fast)  0.675748   6.039729  0.965072        1078.438373
2          XGBoost (T4 GPU)  2.364098   7.239559  0.949817           1.009269
0         Linear Regression  7.174406  26.592481  0.322901           0.258108

Saved results as groundwater_regression_results.csv


# Flood

In [145]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


In [146]:
df = pd.read_csv("flood_model_dataset.csv")

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())

Dataset shape: (6099, 29)

Columns: ['Data Acquisition Time', 'District LGD Code', 'District', 'rainfall', 'river_water_level', 'groundwater_level', 'year', 'month', 'day', 'hour', 'day_of_week', 'season', 'rainfall_lag_1', 'rainfall_lag_3', 'river_lag_1', 'river_lag_3', 'groundwater_lag_1', 'groundwater_lag_3', 'rainfall_roll_3', 'rainfall_roll_6', 'river_roll_3', 'river_roll_6', 'groundwater_roll_3', 'groundwater_roll_6', 'rainfall_diff_1', 'river_diff_1', 'groundwater_diff_1', 'flood_risk_score', 'flood_risk_category']


In [147]:
X = df.drop(columns=["flood_risk_category", "flood_risk_score", "Data Acquisition Time"])
y_raw = df["flood_risk_category"]

# Encode target
le = LabelEncoder()
y = le.fit_transform(y_raw)  # e.g., Low=0, Medium=1, High=2
print("\nEncoded target classes:", le.classes_)


Encoded target classes: ['High' 'Low' 'Medium']


In [148]:
leak_cols = [
    "river_roll_3", "river_roll_6",
    "river_lag_1", "river_lag_3",
    "groundwater_roll_3", "groundwater_roll_6",
    "groundwater_lag_1", "groundwater_lag_3"
]
X = X.drop(columns=leak_cols)

# Identify categorical and numerical columns
categorical_cols = ["District", "season"]
numerical_cols = [col for col in X.columns if col not in categorical_cols]


In [149]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

In [151]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)



Train shape: (4879, 18)
Test shape: (1220, 18)


In [152]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),

    "Random Forest (CPU fast)": RandomForestClassifier(
        n_estimators=300,
        n_jobs=-1,
        random_state=42
    ),

    "XGBoost (GPU)": XGBClassifier(
        tree_method="hist",
        predictor="gpu_predictor",
        device="cuda",
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        use_label_encoder=False,
        eval_metric="mlogloss"
    )
}


In [153]:
results = []

for name, model in models.items():
    print(f"\n===== {name} =====")

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    start = time.time()
    pipeline.fit(X_train, y_train)
    train_time = time.time() - start

    y_pred = pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted")
    rec = recall_score(y_test, y_pred, average="weighted")
    f1 = f1_score(y_test, y_pred, average="weighted")

    results.append([name, acc, prec, rec, f1, train_time])

    print(f"Training Time: {train_time:.2f} sec")
    print("Accuracy :", acc)
    print("Precision:", prec)
    print("Recall   :", rec)
    print("F1 Score :", f1)



===== Logistic Regression =====
Training Time: 0.47 sec
Accuracy : 0.9901639344262295
Precision: 0.9902097217092425
Recall   : 0.9901639344262295
F1 Score : 0.9901374160153652

===== Random Forest (CPU fast) =====
Training Time: 3.16 sec
Accuracy : 0.9950819672131147
Precision: 0.9950941105039466
Recall   : 0.9950819672131147
F1 Score : 0.9950840451115675

===== XGBoost (GPU) =====


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [12:34:55] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "predictor", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Training Time: 1.84 sec
Accuracy : 0.9991803278688525
Precision: 0.9991823617947362
Recall   : 0.9991803278688525
F1 Score : 0.9991803278688525


In [154]:
results_df = pd.DataFrame(
    results,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1 Score", "Training Time (s)"]
).sort_values("F1 Score", ascending=False)

print("\n===== FINAL CLASSIFICATION RESULTS =====")
print(results_df)

# Save results
results_df.to_csv("flood_classification_results.csv", index=False)
print("\nSaved results as flood_classification_results.csv")


===== FINAL CLASSIFICATION RESULTS =====
                      Model  Accuracy  Precision    Recall  F1 Score  \
2             XGBoost (GPU)  0.999180   0.999182  0.999180  0.999180   
1  Random Forest (CPU fast)  0.995082   0.995094  0.995082  0.995084   
0       Logistic Regression  0.990164   0.990210  0.990164  0.990137   

   Training Time (s)  
2           1.844932  
1           3.163147  
0           0.465238  

Saved results as flood_classification_results.csv
